#Bronze Layer

In [0]:
# dbutils.fs.ls("abfss://raw@sthealthcarekl.dfs.core.windows.net/fhir/")

### Schema Creation

In [0]:
%sql
create schema if not exists db_healthcare_kl.raw;
create schema if not exists db_healthcare_kl.bronze;

In [0]:
%sql
SHOW SCHEMAS in db_healthcare_kl


In [0]:
rawSourcePath = "abfss://raw@sthealthcarekl.dfs.core.windows.net/fhir/"

In [0]:
df = spark.read.option("multiline","true").json(rawSourcePath + "*.json")

df.show(5)

In [0]:
df.printSchema()

In [0]:
df.count()

### JSON TO DELTA

In [0]:
%sql
CREATE TABLE if not EXISTS db_healthcare_kl.bronze.fhir_raw
USING DELTA
LOCATION 'abfss://bronze@sthealthcarekl.dfs.core.windows.net/fhir_raw';

In [0]:
%sql
SHOW TABLES IN db_healthcare_kl.bronze;

### Auto Loader

In [0]:
raw_path = "abfss://raw@sthealthcarekl.dfs.core.windows.net/fhir"

schema_path = "abfss://bronze@sthealthcarekl.dfs.core.windows.net/schema/fhir"

checkpoint_path = "abfss://bronze@sthealthcarekl.dfs.core.windows.net/checkpoints/fhir"

bronze_output_path = "abfss://bronze@sthealthcarekl.dfs.core.windows.net/fhir_raw"

In [0]:
schema = (
    spark.read
    .option("multiLine","true")
    .json(raw_path)
    .schema
)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .schema(schema)
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(raw_path)
    .withColumn(
        "ingestion_time",
        current_timestamp()
    )
)

In [0]:

bronze_df.writeStream\
        .format("delta")\
        .option(
            "checkpointLocation",
            checkpoint_path
        )\
        .trigger(availableNow=True)\
        .start(bronze_output_path)


In [0]:
bronze_table = spark.read.table("bronze.fhir_raw")

bronze_table.printSchema()

In [0]:
%sql
DESCRIBE DETAIL db_healthcare_kl.bronze.fhir_raw;

In [0]:
%sql
DESCRIBE HISTORY db_healthcare_kl.bronze.fhir_raw;